<a href="https://colab.research.google.com/github/jshingala/FlashAttention/blob/main/trition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Environment check

In [9]:
import torch
import triton

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"Triton version: {triton.__version__}")


CUDA available: True
Device: Tesla T4
Triton version: 3.6.0


# Copy kernel
Simplest possible Triton kernel: load a block, store it back. Establishes program-id, block offsets, and masking.


In [10]:
import torch
import triton
import triton.language as tl

@triton.jit
def copy_kernel(x_ptr, out_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(axis=0)
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    x = tl.load(x_ptr + offsets, mask=mask)
    tl.store(out_ptr + offsets, x, mask=mask)

def copy(x: torch.Tensor):
    out = torch.empty_like(x)
    n_elements = x.numel()
    BLOCK_SIZE = 1024
    grid = (triton.cdiv(n_elements, BLOCK_SIZE),)
    copy_kernel[grid](x, out, n_elements, BLOCK_SIZE=BLOCK_SIZE)
    return out

x = torch.randn(10000, device="cuda")
out = copy(x)
print("Max difference from input:", (out - x).abs().max().item())
print("Match:", torch.allclose(out, x))


Max difference from input: 0.0
Match: True


#Softmax kernel
One program per row. Stable softmax (subtract the max), padding columns loaded as -inf so exp(-inf)=0 and they can't contribute to the sum.

In [11]:
import torch
import triton
import triton.language as tl

@triton.jit
def softmax_kernel(x_ptr, out_ptr, row_stride, n_cols, BLOCK_SIZE: tl.constexpr):
    row_idx = tl.program_id(axis=0)
    col_offsets = tl.arange(0, BLOCK_SIZE)
    ptrs = x_ptr + row_idx * row_stride + col_offsets
    mask = col_offsets < n_cols
    row = tl.load(ptrs, mask=mask, other=float("-inf"))
    row_minus_max = row - tl.max(row, axis=0)
    numerator = tl.exp(row_minus_max)
    denominator = tl.sum(numerator, axis=0)
    softmax_out = numerator / denominator
    out_ptrs = out_ptr + row_idx * row_stride + col_offsets
    tl.store(out_ptrs, softmax_out, mask=mask)

def softmax(x: torch.Tensor):
    n_rows, n_cols = x.shape
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    out = torch.empty_like(x)
    grid = (n_rows,)
    softmax_kernel[grid](x, out, x.stride(0), n_cols, BLOCK_SIZE=BLOCK_SIZE)
    return out

x = torch.randn(1823, 781, device="cuda")
out_triton = softmax(x)
out_torch = torch.softmax(x, dim=-1)
print("Max difference:", (out_triton - out_torch).abs().max().item())
print("Match:", torch.allclose(out_triton, out_torch, atol=1e-5))


Max difference: 7.450580596923828e-09
Match: True


#Flash attention (online softmax)
The whole point: never materialize the NxN scores matrix. Walk K/V in blocks, keep a running max `m`, running sum `l`, running output `acc`, and re-base the history by `alpha = exp(m_old - m_new)` each time a block raises the max.

In [12]:
import torch
import triton
import triton.language as tl

@triton.jit
def flash_attention_kernel(
    q_ptr, k_ptr, v_ptr, out_ptr,
    stride_qh, stride_qm, stride_qd,
    stride_kh, stride_kn, stride_kd,
    stride_vh, stride_vn, stride_vd,
    stride_oh, stride_om, stride_od,
    N, scale,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, HEAD_DIM: tl.constexpr,
):
    pid_m = tl.program_id(axis=0)
    pid_h = tl.program_id(axis=1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, HEAD_DIM)

    q_ptrs = q_ptr + pid_h * stride_qh + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qd
    q_mask = offs_m[:, None] < N
    q = tl.load(q_ptrs, mask=q_mask, other=0.0)

    m_i = tl.full((BLOCK_M,), float("-inf"), dtype=tl.float32)
    l_i = tl.zeros((BLOCK_M,), dtype=tl.float32)
    acc = tl.zeros((BLOCK_M, HEAD_DIM), dtype=tl.float32)

    for start_n in range(0, N, BLOCK_N):
        offs_n = start_n + tl.arange(0, BLOCK_N)
        k_ptrs = k_ptr + pid_h * stride_kh + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
        v_ptrs = v_ptr + pid_h * stride_vh + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vd
        kv_mask = offs_n[:, None] < N
        k = tl.load(k_ptrs, mask=kv_mask, other=0.0)
        v = tl.load(v_ptrs, mask=kv_mask, other=0.0)

        s = tl.dot(q, tl.trans(k)) * scale
        s = tl.where(offs_n[None, :] < N, s, float("-inf"))

        m_new = tl.maximum(m_i, tl.max(s, axis=1))
        alpha = tl.exp(m_i - m_new)
        p     = tl.exp(s - m_new[:, None])
        l_i   = l_i * alpha + tl.sum(p, axis=1)
        acc   = acc * alpha[:, None] + tl.dot(p.to(v.dtype), v)
        m_i   = m_new

    acc = acc / l_i[:, None]
    out_ptrs = out_ptr + pid_h * stride_oh + offs_m[:, None] * stride_om + offs_d[None, :] * stride_od
    tl.store(out_ptrs, acc, mask=q_mask)

def flash_attention(q, k, v):
    B, H, N, D = q.shape
    q = q.reshape(B * H, N, D); k = k.reshape(B * H, N, D); v = v.reshape(B * H, N, D)
    out = torch.empty_like(q)
    scale = 1.0 / (D ** 0.5)
    BLOCK_M, BLOCK_N = 64, 64
    grid = (triton.cdiv(N, BLOCK_M), B * H)
    flash_attention_kernel[grid](
        q, k, v, out,
        q.stride(0), q.stride(1), q.stride(2),
        k.stride(0), k.stride(1), k.stride(2),
        v.stride(0), v.stride(1), v.stride(2),
        out.stride(0), out.stride(1), out.stride(2),
        N, scale, BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, HEAD_DIM=D,
    )
    return out.reshape(B, H, N, D)

torch.manual_seed(0)
B, H, N, D = 2, 4, 512, 64
q = torch.randn(B, H, N, D, device="cuda")
k = torch.randn(B, H, N, D, device="cuda")
v = torch.randn(B, H, N, D, device="cuda")
out_triton = flash_attention(q, k, v)
scores = (q @ k.transpose(-2, -1)) / (D ** 0.5)
out_torch = torch.softmax(scores, dim=-1) @ v
print("Max difference:", (out_triton - out_torch).abs().max().item())
print("Match:", torch.allclose(out_triton, out_torch, atol=1e-2))


Max difference: 4.76837158203125e-07
Match: True


#Benchmark (memory is the headline)
Naive attention materializes the NxN scores tensor -> peak memory grows O(N^2). Flash stays O(N). The punchline row is where naive OOMs and flash sails through.

**Harness fixes vs first draft:** fewer timing iters (the flash kernel is slow pre-Stage-5, so 100 iters x 547ms was minutes per row and got interrupted); naive timing skipped past N=4096 (its *memory* is the point, its *speed* at huge N isn't, its *death* is); `peak_mem_mb` returns `None` on OOM (not a tuple).


In [13]:
import torch
import triton

def time_ms(fn, *args, warmup=5, iters=20):
    """Median GPU time in ms. CUDA events because launches are async (a CPU
    timer measures launch overhead, not compute); warmup excluded; median."""
    for _ in range(warmup):
        fn(*args)
    torch.cuda.synchronize()
    ts = []
    for _ in range(iters):
        s = torch.cuda.Event(enable_timing=True)
        e = torch.cuda.Event(enable_timing=True)
        s.record(); fn(*args); e.record(); e.synchronize()
        ts.append(s.elapsed_time(e))
    ts.sort()
    return ts[len(ts) // 2]

def peak_mem_mb(fn, *args):
    """Peak CUDA memory (MB) for one call. Returns None if the call OOMs."""
    torch.cuda.synchronize(); torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    base = torch.cuda.memory_allocated()
    try:
        out = fn(*args); torch.cuda.synchronize()
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); return None
    peak = torch.cuda.max_memory_allocated()
    del out
    return (peak - base) / 1e6

def naive_attention(q, k, v):
    scale = 1.0 / (q.shape[-1] ** 0.5)
    scores = (q @ k.transpose(-2, -1)) * scale   # [B,H,N,N] -- the O(N^2) tensor
    return torch.softmax(scores, dim=-1) @ v

def sdpa(q, k, v):
    return torch.nn.functional.scaled_dot_product_attention(q, k, v)

def flops_attention(B, H, N, D):
    return 4.0 * B * H * N * N * D

def run(dtype=torch.float16):
    if not torch.cuda.is_available():
        raise SystemExit("No CUDA device.")
    torch.manual_seed(0)
    configs = [(1,16,512,64),(1,16,1024,64),(1,16,2048,64),
               (1,16,4096,64),(1,16,8192,64),(1,16,16384,64)]
    print(f"dtype={dtype}")
    print(f"{'N':>7} | {'corr':>5} | {'naive MB':>10} | {'flash MB':>10} | {'sdpa MB':>10} | "
          f"{'naive ms':>9} | {'flash ms':>9} | {'sdpa ms':>9} | {'flash TFLOP/s':>13}")
    print("-" * 110)
    for (B, H, N, D) in configs:
        q = torch.randn(B,H,N,D, device="cuda", dtype=dtype)
        k = torch.randn(B,H,N,D, device="cuda", dtype=dtype)
        v = torch.randn(B,H,N,D, device="cuda", dtype=dtype)
        try:
            out_t = flash_attention(q,k,v)
            out_ref = naive_attention(q,k,v)
            atol = 1e-2 if dtype==torch.float32 else 2e-2
            corr = "ok" if torch.allclose(out_t, out_ref, atol=atol) else "FAIL"
            del out_ref
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); corr = "n/a"
        m_naive = peak_mem_mb(naive_attention, q,k,v)
        m_flash = peak_mem_mb(flash_attention, q,k,v)
        m_sdpa  = peak_mem_mb(sdpa, q,k,v)
        # Skip timing naive past 4096: its memory (above) is the point; timing a
        # huge slow O(N^2) call just burns minutes. Its OOM is the punchline.
        if m_naive is None:
            t_naive = "OOM"
        elif N > 4096:
            t_naive = "skip"
        else:
            t_naive = f"{time_ms(naive_attention, q,k,v):.3f}"
        t_flash = time_ms(flash_attention, q,k,v)
        t_sdpa  = time_ms(sdpa, q,k,v)
        tflops_flash = flops_attention(B,H,N,D) / (t_flash*1e-3) / 1e12
        fmt = lambda m: "OOM" if m is None else f"{m:.1f}"
        print(f"{N:>7} | {corr:>5} | {fmt(m_naive):>10} | {fmt(m_flash):>10} | {fmt(m_sdpa):>10} | "
              f"{t_naive:>9} | {t_flash:>9.3f} | {t_sdpa:>9.3f} | {tflops_flash:>13.1f}")
        del q, k, v; torch.cuda.empty_cache()

run(dtype=torch.float16)


dtype=torch.float16
      N |  corr |   naive MB |   flash MB |    sdpa MB |  naive ms |  flash ms |   sdpa ms | flash TFLOP/s
--------------------------------------------------------------------------------------------------------------
    512 |    ok |       17.8 |        1.0 |        1.0 |     0.391 |     2.908 |     0.252 |           0.4
   1024 |    ok |       69.2 |        2.1 |        2.1 |     1.161 |     8.865 |     0.352 |           0.5
   2048 |    ok |      272.6 |        4.2 |        4.2 |     4.201 |    34.595 |     1.171 |           0.5
   4096 |    ok |     1082.1 |        8.4 |        8.4 |    20.025 |   137.051 |     6.001 |           0.5
   8192 |    ok |     4311.7 |       16.8 |       16.8 |      skip |   553.927 |    23.372 |           0.5
  16384 |   n/a |        OOM |       33.6 |       33.6 |       OOM |  2174.210 |    88.397 |           0.5


# Make it fast (autotuning)
Stage 4 proved the **memory** win. But flash sat at a flat **0.5 TFLOP/s** at every size — the signature of one under-powered launch config (64x64, default warps/stages) leaving the tensor cores idle. The kernel *body* is unchanged and already correct; the only new thing is `@triton.autotune`, which benchmarks several `(BLOCK_M, BLOCK_N, num_warps, num_stages)` configs once per shape and caches the winner. Bigger tiles feed the tensor cores; `num_stages>=2` pipelines K/V loads to hide HBM latency.

**First call per shape is slow** (autotuner is benchmarking every config) — that cost is paid once, then cached. Re-run the benchmark cell below afterward to see the real throughput.

In [8]:
import torch
import triton
import triton.language as tl


# ---------------------------------------------------------------------------
# Stage 5: SAME kernel body as Stage 3 (the online-softmax recurrence is
# already correct). The ONLY new thing is @triton.autotune: Triton benchmarks
# the configs below once per unique problem shape, caches the winner, and
# launches with it. This is what fixes the flat 0.5 TFLOP/s — that flatness
# was a single under-powered launch config (64x64, default warps/stages)
# leaving the tensor cores idle, not a flaw in the math.
#
# Why these knobs:
#   BLOCK_M / BLOCK_N : bigger tiles = more work per step on the tensor cores,
#                       less loop + memory-latency overhead. 64x64 was too small.
#   num_warps         : threads per program. More warps = more parallelism to
#                       hide memory latency, up to a point (then register
#                       pressure hurts). Let autotune find the knee.
#   num_stages        : software pipelining depth. >=2 lets the NEXT block's
#                       K/V load overlap with THIS block's compute. The single
#                       biggest lever for hiding HBM latency on a T4.
# HEAD_DIM stays a kwarg (it's fixed by the data, not a thing to search).
# ---------------------------------------------------------------------------

def _configs():
    cfgs = []
    for bm in (64, 128):
        for bn in (32, 64, 128):
            for w in (2, 4, 8):
                for s in (2, 3, 4):
                    cfgs.append(
                        triton.Config(
                            {"BLOCK_M": bm, "BLOCK_N": bn},
                            num_warps=w, num_stages=s,
                        )
                    )
    return cfgs


@triton.autotune(configs=_configs(), key=["N", "HEAD_DIM"])
@triton.jit
def flash_attention_kernel(
    q_ptr, k_ptr, v_ptr, out_ptr,
    stride_qh, stride_qm, stride_qd,
    stride_kh, stride_kn, stride_kd,
    stride_vh, stride_vn, stride_vd,
    stride_oh, stride_om, stride_od,
    N,                              # sequence length
    scale,                          # 1 / sqrt(head_dim)
    BLOCK_M: tl.constexpr,          # query rows per program (autotuned)
    BLOCK_N: tl.constexpr,          # key/value rows per inner step (autotuned)
    HEAD_DIM: tl.constexpr,         # head dimension D (fixed by data)
):
    pid_m = tl.program_id(axis=0)
    pid_h = tl.program_id(axis=1)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, HEAD_DIM)

    q_ptrs = q_ptr + pid_h * stride_qh \
        + offs_m[:, None] * stride_qm + offs_d[None, :] * stride_qd
    q_mask = offs_m[:, None] < N
    q = tl.load(q_ptrs, mask=q_mask, other=0.0)

    # running state — unchanged from Stage 3
    m_i = tl.full((BLOCK_M,), float("-inf"), dtype=tl.float32)
    l_i = tl.zeros((BLOCK_M,), dtype=tl.float32)
    acc = tl.zeros((BLOCK_M, HEAD_DIM), dtype=tl.float32)

    for start_n in range(0, N, BLOCK_N):
        offs_n = start_n + tl.arange(0, BLOCK_N)

        k_ptrs = k_ptr + pid_h * stride_kh \
            + offs_n[:, None] * stride_kn + offs_d[None, :] * stride_kd
        v_ptrs = v_ptr + pid_h * stride_vh \
            + offs_n[:, None] * stride_vn + offs_d[None, :] * stride_vd
        kv_mask = offs_n[:, None] < N
        k = tl.load(k_ptrs, mask=kv_mask, other=0.0)
        v = tl.load(v_ptrs, mask=kv_mask, other=0.0)

        # scores — unchanged
        s = tl.dot(q, tl.trans(k)) * scale
        s = tl.where(offs_n[None, :] < N, s, float("-inf"))

        # online-softmax recurrence — unchanged (this was always correct)
        m_new = tl.maximum(m_i, tl.max(s, axis=1))
        alpha = tl.exp(m_i - m_new)
        p     = tl.exp(s - m_new[:, None])
        l_i   = l_i * alpha + tl.sum(p, axis=1)
        acc   = acc * alpha[:, None] + tl.dot(p.to(v.dtype), v)
        m_i   = m_new

    acc = acc / l_i[:, None]

    out_ptrs = out_ptr + pid_h * stride_oh \
        + offs_m[:, None] * stride_om + offs_d[None, :] * stride_od
    tl.store(out_ptrs, acc, mask=q_mask)


def flash_attention(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor):
    B, H, N, D = q.shape
    q = q.reshape(B * H, N, D)
    k = k.reshape(B * H, N, D)
    v = v.reshape(B * H, N, D)
    out = torch.empty_like(q)

    scale = 1.0 / (D ** 0.5)
    # NOTE: BLOCK_M/BLOCK_N are no longer passed here — autotune supplies them.
    # The grid uses a META lambda so it's computed from whichever BLOCK_M the
    # autotuner picks for this shape.
    grid = lambda META: (triton.cdiv(N, META["BLOCK_M"]), B * H)

    flash_attention_kernel[grid](
        q, k, v, out,
        q.stride(0), q.stride(1), q.stride(2),
        k.stride(0), k.stride(1), k.stride(2),
        v.stride(0), v.stride(1), v.stride(2),
        out.stride(0), out.stride(1), out.stride(2),
        N, scale,
        HEAD_DIM=D,
    )
    return out.reshape(B, H, N, D)


# --- correctness check (unchanged from Stage 3) ---
if __name__ == "__main__":
    torch.manual_seed(0)
    B, H, N, D = 2, 4, 512, 64
    q = torch.randn(B, H, N, D, device="cuda", dtype=torch.float16)
    k = torch.randn(B, H, N, D, device="cuda", dtype=torch.float16)
    v = torch.randn(B, H, N, D, device="cuda", dtype=torch.float16)

    out_triton = flash_attention(q, k, v)
    scores = (q.float() @ k.float().transpose(-2, -1)) / (D ** 0.5)
    out_torch = (torch.softmax(scores, dim=-1) @ v.float()).to(torch.float16)

    print("Max difference:", (out_triton.float() - out_torch.float()).abs().max().item())
    print("Match:", torch.allclose(out_triton.float(), out_torch.float(), atol=2e-2))

Max difference: 0.000244140625
Match: True


### Re-run the benchmark with the autotuned kernel
The `flash_attention` defined in Stage 5 now shadows the Stage 3 one, so the Stage 4 `run()` will use the fast kernel. (Re-run the Stage 4 cell, or this copy.)

In [14]:
run(dtype=torch.float16)


dtype=torch.float16
      N |  corr |   naive MB |   flash MB |    sdpa MB |  naive ms |  flash ms |   sdpa ms | flash TFLOP/s
--------------------------------------------------------------------------------------------------------------
    512 |    ok |       17.8 |        1.0 |        1.0 |     0.400 |     2.981 |     0.261 |           0.4
   1024 |    ok |       69.2 |        2.1 |        2.1 |     1.303 |     9.216 |     0.363 |           0.5
   2048 |    ok |      272.6 |        4.2 |        4.2 |     4.183 |    34.687 |     1.274 |           0.5
   4096 |    ok |     1082.1 |        8.4 |        8.4 |    20.164 |   136.980 |     5.999 |           0.5
   8192 |    ok |     4311.7 |       16.8 |       16.8 |      skip |   552.706 |    23.282 |           0.5
  16384 |   n/a |        OOM |       33.6 |       33.6 |       OOM |  2169.883 |    88.861 |           0.5
